In [ ]:
"""
ONE-TIME historical workflow (replaces both separate scripts):

A) Earthaccess
   - Login using .netrc
   - Find DAWG SOS discharge NetCDF for Europe 
   - Download to local folder

B) Hydrocron
   - Load reach_ids from a GeoPackage layer
   - For each reach_id, download RiverSP timeseries (WSE, width, slope, etc.)

C) Merge
   - Parse both to UTC datetimes internally
   - Outer-merge Hydrocron + consensus_q per reach on timestamp
   - Write ONE time column to CSV: time_utc (ISO string: YYYY-MM-DDTHH:MM:SSZ)

Outputs:
  OUT_DIR/reach_<reach_id>.csv
"""

import os, json, time, random
from io import StringIO
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import netCDF4 as nc
import earthaccess


# -----------------------------
# CONFIG
# -----------------------------
GPKG_PATH = r"dnipro_sword_reaches_clip.gpkg"
LAYER_NAME = "dnipro_reaches"
REACH_ID_FIELD = "reach_id"

HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_2.0"
FIELDS = "time_str,cycle_id,pass_id,wse,slope,width"

START_TIME = "2022-12-01T00:00:00Z"
END_TIME = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

SHORT_NAME = "SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3"
DOWNLOAD_DIR = "./downloaded_files"

OUT_DIR = r"riverSP_DAWG_reach"
os.makedirs(OUT_DIR, exist_ok=True)

TIMEOUT_S = 60
SLEEP_BETWEEN_REQUESTS_S = (0.2, 0.6)
MAX_RETRIES = 5


# -----------------------------
# Earthaccess: login + download
# -----------------------------
def download_dawg_sos_europe_netcdf() -> str:
    earthaccess.login(strategy="netrc")
    print("Authenticated with Earthaccess using netrc")

    print(f"Searching granules for {SHORT_NAME} ...")
    granules = earthaccess.search_data(short_name=SHORT_NAME, count=500)

    eu_granules = [g for g in granules if g["meta"]["native-id"].startswith("eu_")]
    if not eu_granules:
        raise RuntimeError("No Europe DAWG SOS granules found (native-id starting with 'eu_').")

    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    paths = earthaccess.download(eu_granules, local_path=DOWNLOAD_DIR)
    if not paths:
        raise RuntimeError("earthaccess.download returned no files.")

    # If multiple EU files, pick "newest" by filename (often includes timestamps)
    paths_sorted = sorted(paths, key=lambda p: os.path.basename(p))
    nc_path = paths_sorted[-1]
    print("Downloaded EU DAWG SOS NetCDF(s):", len(paths))
    print("Using NetCDF:", nc_path)
    return nc_path


# -----------------------------
# Hydrocron helpers
# -----------------------------
def hydrocron_response_to_df(text: str) -> pd.DataFrame:
    text = (text or "").strip()
    if not text:
        return pd.DataFrame()
    if text.startswith("{"):
        try:
            obj = json.loads(text)
        except json.JSONDecodeError:
            return pd.DataFrame()
        csv_text = (obj.get("results", {}).get("csv", "") or "").strip()
        return pd.read_csv(StringIO(csv_text)) if csv_text else pd.DataFrame()
    return pd.read_csv(StringIO(text))


def request_with_retries(url: str, params: dict) -> requests.Response:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, params=params, timeout=TIMEOUT_S)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}")
            return r
        except Exception as e:
            last_err = e
            backoff = min(30, (2 ** (attempt - 1)) * 0.7) + random.uniform(0, 0.5)
            time.sleep(backoff)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err}")


def safe_reach_filename(reach_id: str) -> str:
    s = str(reach_id).strip()
    cleaned = "".join(ch for ch in s if ch.isalnum() or ch in ("-", "_"))
    return cleaned or "unknown_reach"


def load_reach_ids_from_gpkg() -> list[str]:
    gdf = gpd.read_file(GPKG_PATH, layer=LAYER_NAME)
    if REACH_ID_FIELD not in gdf.columns:
        raise ValueError(f"Field '{REACH_ID_FIELD}' not found in layer '{LAYER_NAME}'")
    s = (
        gdf[REACH_ID_FIELD]
        .dropna()
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    return s.loc[s != ""].unique().tolist()


def hydrocron_fetch_reach_df(reach_id: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "start_time": start_iso,
        "end_time": end_iso,
        "output": "csv",
        "collection_name": COLLECTION_NAME,
        "fields": FIELDS,
    }
    time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS_S))
    r = request_with_retries(HYDROCRON_URL, params)
    df = hydrocron_response_to_df(r.text)

    # Parse Hydrocron time_str like 2023-04-06T21:16:48Z to UTC datetime for merging
    if not df.empty and "time_str" in df.columns:
        df["_dt"] = pd.to_datetime(df["time_str"], errors="coerce", utc=True)
    else:
        df["_dt"] = pd.to_datetime(pd.Series([], dtype="datetime64[ns]"), utc=True)

    df["reach_id"] = str(reach_id)
    df = df.dropna(subset=["_dt"])

    preferred = ["reach_id", "_dt", "cycle_id", "pass_id", "wse", "slope", "width"]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred and c != "time_str"]
    return df[cols]


# -----------------------------
# NetCDF helpers (consensus_q)
# -----------------------------
def open_dawg_sos(nc_path: str):
    ds = nc.Dataset(nc_path, "r")
    reaches = ds.groups["reaches"]
    consensus = ds.groups["consensus"]

    nc_reach_ids = reaches.variables["reach_id"][:].astype("int64")
    id_to_idx = {int(rid): int(i) for i, rid in enumerate(nc_reach_ids)}

    qvar = consensus.variables["consensus_q"]
    time_var = consensus.variables["time_int"]

    missing = None
    if "_FillValue" in qvar.ncattrs():
        missing = qvar.getncattr("_FillValue")
    elif "missing_value" in qvar.ncattrs():
        missing = qvar.getncattr("missing_value")

    return ds, id_to_idx, time_var, qvar, missing


def consensus_q_df_for_reach(rid_int: int, id_to_idx, time_var, qvar, missing) -> pd.DataFrame:
    if rid_int not in id_to_idx:
        return pd.DataFrame(columns=["reach_id", "_dt", "consensus_q"])

    i = id_to_idx[rid_int]

    times = np.asarray(time_var[i], dtype="float64")
    valid_time = times > -9.0e10
    times_valid = times[valid_time].astype("int64")

    if times_valid.size == 0:
        return pd.DataFrame(columns=["reach_id", "_dt", "consensus_q"])

    # seconds since 2000-01-01 00:00:00 (treat as UTC)
    dt64 = np.array([np.datetime64("2000-01-01T00:00:00") + np.timedelta64(int(t), "s") for t in times_valid])
    dt = pd.to_datetime(dt64.astype("datetime64[ns]"), utc=True)

    q = np.asarray(qvar[i], dtype="float64")[valid_time]
    if missing is not None:
        q[q == missing] = np.nan
    q[q <= -9.0e10] = np.nan

    return pd.DataFrame({"reach_id": rid_int, "_dt": dt, "consensus_q": q}).dropna(subset=["_dt"])


# -----------------------------
# Merge + finalize one time column
# -----------------------------
def merge_on_time(hydro_df: pd.DataFrame, q_df: pd.DataFrame) -> pd.DataFrame:
    hydro_df["_dt"] = pd.to_datetime(hydro_df.get("_dt"), errors="coerce", utc=True)
    q_df["_dt"] = pd.to_datetime(q_df.get("_dt"), errors="coerce", utc=True)

    merged = pd.merge(
        hydro_df,
        q_df[["_dt", "consensus_q"]],
        on="_dt",
        how="outer",
    )

    # fill reach_id for consensus_q-only rows
    if "reach_id" in merged.columns and merged["reach_id"].isna().any():
        nonnull = merged["reach_id"].dropna()
        if len(nonnull):
            merged["reach_id"] = merged["reach_id"].fillna(nonnull.iloc[0])

    merged = merged.sort_values("_dt")
    merged = merged.drop_duplicates(subset=["_dt"], keep="first")

    # Final single time column (ISO UTC with Z)
    merged["time_utc"] = pd.to_datetime(merged["_dt"], utc=True, errors="coerce").dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    # Drop internal datetime key
    merged = merged.drop(columns=["_dt"], errors="ignore")

    # Order columns
    cols = list(merged.columns)
    front = ["reach_id", "time_utc"]
    ordered = [c for c in front if c in cols] + [c for c in cols if c not in front]
    return merged[ordered]


def main():
    nc_path = download_dawg_sos_europe_netcdf()
    reach_ids = load_reach_ids_from_gpkg()

    print("\nHistorical Combined Download (Hydrocron + consensus_q)")
    print("Reaches:", len(reach_ids))
    print("Window:", START_TIME, "→", END_TIME)
    print("OUT_DIR:", OUT_DIR)

    ds, id_to_idx, time_var, qvar, missing = open_dawg_sos(nc_path)

    try:
        for n, rid in enumerate(reach_ids, 1):
            rid_str = str(rid).strip()
            try:
                rid_int = int(float(rid_str))  # handles "123.0"
            except Exception:
                rid_int = None

            out_csv = os.path.join(OUT_DIR, f"reach_{safe_reach_filename(rid_str)}.csv")

            try:
                hydro_df = hydrocron_fetch_reach_df(rid_str, START_TIME, END_TIME)

                if rid_int is None:
                    q_df = pd.DataFrame(columns=["reach_id", "_dt", "consensus_q"])
                else:
                    q_df = consensus_q_df_for_reach(rid_int, id_to_idx, time_var, qvar, missing)

                merged = merge_on_time(hydro_df, q_df)
                merged.to_csv(out_csv, index=False)

                print(
                    f"[{n}/{len(reach_ids)}] {rid_str} "
                    f"hydro={len(hydro_df)} "
                    f"discharge={len(q_df)} "
                    f"merged={len(merged)}"
                )

            except Exception as e:
                print(f"[{n}/{len(reach_ids)}] {rid_str} ERROR: {e}")

    finally:
        ds.close()

    print("\nDone.")


if __name__ == "__main__":
    main()

Authenticated with Earthaccess using netrc
Searching granules for SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3 ...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Downloaded EU DAWG SOS NetCDF(s): 1
Using NetCDF: downloaded_files/eu_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc

Historical Combined Download (Hydrocron + consensus_q)
Reaches: 842
Window: 2022-12-01T00:00:00Z → 2026-03-05T00:01:27Z
OUT_DIR: riverSP_DAWG_reach
[1/842] 22511300103 hydro=169 discharge=0 merged=169
[2/842] 22400900035 hydro=187 discharge=0 merged=187
[3/842] 22601000045 hydro=0 discharge=0 merged=0
[4/842] 22511100015 hydro=0 discharge=0 merged=0
[5/842] 22511100055 hydro=154 discharge=0 merged=154
[6/842] 22511100021 hydro=157 discharge=0 merged=157
[7/842] 22511100031 hydro=166 discharge=0 merged=166
[8/842] 22511100041 hydro=0 discharge=0 merged=0
[9/842] 22511300011 hydro=166 discharge=0 merged=166
[10/842] 22511300291 hydro=0 discharge=0 merged=0
[11/842] 22511300021 hydro=167 discharge=143 merged=167
[12/842] 22511300281 hydro=0 discharge=0 merged=0
[13/842] 22511200011 hydro=166 discharge=120 merged=166
[14/842] 22511300

/tmp/ipykernel_3814471/3540101772.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["_dt"] = pd.to_datetime(df["time_str"], errors="coerce", utc=True)


[30/842] 22511300263 hydro=84 discharge=0 merged=84
[31/842] 22511300123 hydro=0 discharge=0 merged=0
[32/842] 22511300243 hydro=84 discharge=0 merged=84
[33/842] 22511300256 hydro=0 discharge=0 merged=0
[34/842] 22511300133 hydro=118 discharge=0 merged=118
[35/842] 22511300143 hydro=43 discharge=0 merged=43
[36/842] 22511300153 hydro=48 discharge=0 merged=48


/tmp/ipykernel_3814471/3540101772.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["_dt"] = pd.to_datetime(df["time_str"], errors="coerce", utc=True)


[37/842] 22511300163 hydro=47 discharge=0 merged=47
[38/842] 22511200081 hydro=169 discharge=92 merged=169
[39/842] 22511200091 hydro=198 discharge=127 merged=198
[40/842] 22511200101 hydro=198 discharge=118 merged=198
